In [55]:
%pip install --quiet langchain-groq langchain tqdm openai tiktoken

Note: you may need to restart the kernel to use updated packages.


In [56]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

from sqlalchemy import create_engine
import pandas as pd
import os
from dotenv import load_dotenv
import numpy as np
from sqlalchemy import text
from langchain_groq import ChatGroq
import tiktoken
import openai

In [57]:
load_dotenv()

MYSQL_USER = os.getenv('MYSQL_USER')
MYSQL_PASSWORD = os.getenv('MYSQL_PASSWORD')
MYSQL_DB = os.getenv('MYSQL_DB')
MYSQL_HOST = os.getenv('MYSQL_HOST')
DATABASE_URI = f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}/{MYSQL_DB}'
engine = create_engine(DATABASE_URI)

In [58]:
ENDPOINT = "ANYSCALE" 
# ENDPOINT = "GROK"

CWE = 22

# LLM_MODEL = "gpt-3.5-turbo"
LLM_MODEL = 'gpt-4-turbo'
# LLM_MODEL = "llama3-70b-8192"
# LLM_MODEL = "mixtral-8x7b-32768"

# limit -> 4000 llama3
# 2500 mixtral
# 5th run only for llama with larger dataset
runs = {
    79: 5,
    89: 10,
    22: 20
}
SAFE_FILES = False

RUN = runs[CWE] if not SAFE_FILES else runs[CWE] + 1
ENDPOINT_TOKENS_LIMIT = 8000000
WORKERS = 200


chat = ChatGroq(temperature=0, model_name=LLM_MODEL, groq_api_key=os.getenv('GROK_API_KEY'))

# Select files - all for a given CWE - and double

In [59]:
def select_all_files():
    # lead csv data.csv with pandss
    filename = f'files_nith_{CWE}.csv' if not SAFE_FILES else f'files_nith_safe_{CWE}.csv'
    df = pd.read_csv(filename)
    # when target_bug_pos then it must be divisible by 1000
    df = df[(df['target_bug_pos'] % 2000 == 0) | (df['target_bug_pos'] <= 1000)]
    df = df[((df['target_characters'] % 2000 == 0) & (df['target_characters'] < 38000)) | (df['target_characters'] <= 1000)]
    return df

df_selected_files = select_all_files()
df_selected_files = df_selected_files.sort_values(by='total_length', ascending=True)

# Run inference

In [60]:
def file_exists(file_id, model, run):
    query = text(f"""
    SELECT 
        COUNT(*) AS count
    FROM 
        inference_nith
    WHERE 
        file_id = :file_id
        AND model = :model
        AND run = :run
    """)
    with engine.connect() as connection:
        result = connection.execute(query, {'file_id': file_id, 'model': model, 'run': run})
        row = result.fetchone()
        return row[0] > 0
    
def num_tokens_from_string(string: str, encoding_name = 'cl100k_base') -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens
    
def run_for_file_id(file_id, content, run):

    prompt, response = run_inference(content, CWE)

    # Inserting the inference result into the database
    insert_query = """
    INSERT INTO inference_nith (file_id, prompt, response, model, run)
    VALUES (:file_id, :prompt, :response, :model, :run);
    """
    try:
        with engine.connect() as connection:
            with connection.begin():
                connection.execute(text(insert_query), {
                    'file_id': file_id, 
                    # 'type': "buggy" if is_buggy else "not_buggy",
                    'prompt': prompt,  
                    'response': response,
                    'model': LLM_MODEL,
                    'run': run
                })
    except Exception as e:
        print(f"An error occurred: {e}")
        print("Failed to save analysis to database.")
        return 1

    print("Analysis complete and saved to database.")
    return 0

def get_model_response(prompt, data):
    prompt_whole = prompt.format(**data)

    
    if ENDPOINT== "ANYSCALE":
        if LLM_MODEL == "gpt-3.5-turbo" or LLM_MODEL == "gpt-4-turbo":
            model = LLM_MODEL
            client = openai.OpenAI(
                api_key = os.environ["OPENAI_API_KEY"],
            )
            
        else:
            anyscale_names = {
                "llama3-70b-8192": "meta-llama/Meta-Llama-3-70B-Instruct",
                "mixtral-8x7b-32768": "mistralai/Mixtral-8x7B-Instruct-v0.1"
            }

            model = anyscale_names.get(LLM_MODEL, None)
            if model is None:
                raise ValueError(f"Model {LLM_MODEL} is not supported by anyscale")
            
            client = openai.OpenAI(
                base_url = "https://api.endpoints.anyscale.com/v1",
                api_key = os.environ["ANYSCALE_API_KEY"],
            )

        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt.format(**data)}],
            temperature=0,
            max_tokens=700
        )

        chat_completion = resp.model_dump()
        response = chat_completion["choices"][0]["message"]["content"]
        return (prompt_whole, response) 
    
    elif ENDPOINT == "GROK":
        prompts = ChatPromptTemplate.from_messages([("human", prompt)])

        chain = prompts | chat
        response = chain.invoke(data)
        return (prompt_whole, response.content) #there is no content from anyscale

In [61]:
def run_inference(file_content, cwe_id = "79", github_repo=''):
    cwe_labels = {
        'CWE-79': 'Improper Neutralization of Input During Web Page Generation: Cross-Site Scripting',
        'CWE-89': "Improper Neutralization of Special Elements used in an SQL Command ('SQL Injection')",
        'CWE-22': 'Improper Limitation of a Pathname to a Restricted Directory ("Path Traversal")',
    }
     
    data = {'cwe_type': f"CWE-{cwe_id}", 'file_content': file_content, 'github_repo': github_repo, "cwe_label": cwe_labels[f"CWE-{cwe_id}"]}
    
    prompt = '''Analyze the file content below and tell me if there's any line that may contain a bug of type {cwe_type} ({cwe_label}). Your output must adhere to the following structure.
 
Expected Output Structure:
SE: very Short Explanation of why the line may contain a bug of given type (e.g., The 'user_input' is directly concatenated into HTML content without sanitation).
BL: the Bugged Line, if any is found, else none (e.g., `response = "<html><body><h1>Welcome, " + user_input + "!</h1></body></html>"`).
BUG FOUND: YES if a bug is found, else NO.

Example output:
SE: The 'user_input' is directly concatenated into HTML content without sanitation.
BL: `response = "<html><body><h1>Welcome, " + user_input + "!</h1></body></html>"`
BUG FOUND: YES

File Content:
{file_content}
    ''' 
    return get_model_response(prompt, data)

# run for detected files

In [62]:
import threading
import time
from queue import Queue
from tqdm import tqdm
from threading import Semaphore, Lock


# # Initialize the semaphore with the total number of tokens available per minute
class TokenDecayLimiter:
    def __init__(self, max_tokens, rate_decay):
        self.max_tokens = max_tokens
        self.tokens = max_tokens
        self.rate_decay = rate_decay  # rate per second
        self.last_check = time.time()
        self.lock = Lock()

    def acquire(self, tokens_needed):
        with self.lock:
            current_time = time.time()
            elapsed_time = current_time - self.last_check
            # Increase available tokens based on the time elapsed and rate of decay
            self.tokens = min(self.max_tokens, self.tokens + int(elapsed_time * self.rate_decay))
            self.last_check = current_time

            if tokens_needed <= self.tokens:
                self.tokens -= tokens_needed
                return True
            return False

    def try_acquire(self, tokens_needed):
        while not self.acquire(tokens_needed):
            # print(f"{time.strftime('%Y-%m-%d %H:%M:%S')} Waiting for token availability, current available tokens {self.tokens}...")
            time.sleep(1)  # Wait a bit before trying again
        return True
            
    def release(self, tokens):
        with self.lock:
            self.tokens += tokens

# Worker function remains the same
def worker(runNr, limiter, pbar=None):
    global tokens_used
    while True:
        file = work_queue.get()
        if file is None:  # Stop signal
            work_queue.task_done()
            break

        if file['file_id'] == 124:
            work_queue.task_done()
            continue  # Skip file_id 124

        if file_exists(file['file_id'], LLM_MODEL, runNr):
            if pbar:
                pbar.update(1)
            work_queue.task_done()
            continue
        
        tokens_needed = num_tokens_from_string(file['content']) + 205 #For prompt
        # print with current timepstamp and "Aquiring tokens"
        if limiter.try_acquire(tokens_needed):
        
            try:
                if run_for_file_id(file['file_id'], file['content'], runNr) == 0:
                    print(f"Thread {threading.current_thread().name}: Analysis completed for file_id: {file['file_id']}")
                else:
                    print(f"Thread {threading.current_thread().name}: Analysis failed for file_id: {file['file_id']}")
                    # add the tokens back to the semaphore
                    limiter.release(tokens_needed)
            except Exception as e:
                print(f"Thread {threading.current_thread().name}: Issues with file {file['file_id']}: {str(e)}")
            finally:
                if pbar:
                    pbar.update(1)
                work_queue.task_done()


In [63]:

work_queue = Queue()
for i, file in df_selected_files.iterrows():
    work_queue.put(file)

# Progress bar setup
pbar = tqdm(total=df_selected_files.shape[0])
limiter = TokenDecayLimiter(ENDPOINT_TOKENS_LIMIT, ENDPOINT_TOKENS_LIMIT / 60)
# Start worker threads
threads = []
for _ in range(WORKERS):  # Adjust number based on your concurrency needs
    t = threading.Thread(target=worker, args=(RUN, limiter, pbar))
    t.start()
    threads.append(t)

# Wait for all tasks to be processed
for t in threads:
    work_queue.put(None)  # Signal to threads to stop

for t in threads:
    t.join()

# Ensure the progress bar is closed after all threads complete
pbar.close()
print("All files processed.")

  1%|          | 5/622 [00:06<02:32,  4.05it/s]  

Analysis complete and saved to database.
Thread Thread-1123 (worker): Analysis completed for file_id: 5968
Analysis complete and saved to database.
Thread Thread-1127 (worker): Analysis completed for file_id: 3003
Analysis complete and saved to database.
Thread Thread-1131 (worker): Analysis completed for file_id: 3015
Analysis complete and saved to database.
Thread Thread-1125 (worker): Analysis completed for file_id: 3
Analysis complete and saved to database.
Thread Thread-1122 (worker): Analysis completed for file_id: 3005
Analysis complete and saved to database.
Thread Thread-1130 (worker): Analysis completed for file_id: 5985
Analysis complete and saved to database.
Thread Thread-1129 (worker): Analysis completed for file_id: 5969
Analysis complete and saved to database.
Thread Thread-1136 (worker): Analysis completed for file_id: 17
Analysis complete and saved to database.
Thread Thread-1133 (worker): Analysis completed for file_id: 5980


  2%|▏         | 10/622 [00:07<02:05,  4.89it/s]

Analysis complete and saved to database.
Thread Thread-1134 (worker): Analysis completed for file_id: 15


  2%|▏         | 11/622 [00:07<02:17,  4.45it/s]

...

Analysis complete and saved to database.
Thread Thread-1186 (worker): Analysis completed for file_id: 8080


100%|█████████▉| 621/622 [01:49<00:00,  1.38it/s]

Analysis complete and saved to database.
Thread Thread-1222 (worker): Analysis completed for file_id: 1921


100%|██████████| 622/622 [01:54<00:00,  5.45it/s]

Analysis complete and saved to database.
Thread Thread-1234 (worker): Analysis completed for file_id: 2183
All files processed.
